# Stage 1A: Data Collection from arXiv

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 1.0.0 | 2026-01-26 | That Le | arXiv PDF collection pipeline |

---

## Overview

This notebook handles collecting academic papers (PDFs) from arXiv that likely contain charts and visualizations.

### Pipeline Position

```
[01a_data_collection] --> 01b_image_extraction --> 01c_chart_classification --> 01d_qa_generation
         |
         v
    PDF files in data/raw_pdfs/
```

### Current Status

| Metric | Value |
| --- | --- |
| Current PDFs | ~889 |
| Target PDFs | 10,000 |
| Source | arXiv API |

In [19]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Set to True to actually download PDFs (uses network/disk)
EXECUTE_DOWNLOAD = True  # <-- Change to True to download

# Download settings
TARGET_PDF_COUNT = 10000  # Target total PDFs
BATCH_SIZE = 100          # Papers per batch
RATE_LIMIT_SECONDS = 3    # Delay between downloads

print(f"Download mode: {'ACTIVE' if EXECUTE_DOWNLOAD else 'DRY RUN'}")
print(f"Target: {TARGET_PDF_COUNT} PDFs")

Download mode: ACTIVE
Target: 10000 PDFs


In [20]:
# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================
import sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Data directories
RAW_PDFS_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
PROGRESS_FILE = PROJECT_ROOT / "data" / "arxiv_progress.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"PDFs directory: {RAW_PDFS_DIR}")

Project root: d:\elix\chart_analysis_ai_v3
PDFs directory: d:\elix\chart_analysis_ai_v3\data\raw_pdfs


---

## 1. Current Dataset Statistics

Before downloading more, let's check what we already have.

In [28]:
# ============================================================================
# CHECK CURRENT DATASET
# ============================================================================
import json

def get_current_stats():
    """Get statistics about current PDF collection."""
    pdf_files = list(RAW_PDFS_DIR.glob("*.pdf")) if RAW_PDFS_DIR.exists() else []
    
    # Calculate total size
    total_size_mb = sum(f.stat().st_size for f in pdf_files) / (1024 * 1024)
    
    # Load progress if exists
    progress = {}
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            progress = json.load(f)
    
    return {
        "total_pdfs": len(pdf_files),
        "total_size_mb": round(total_size_mb, 2),
        "avg_size_mb": round(total_size_mb / len(pdf_files), 2) if pdf_files else 0,
        "downloaded": len(progress.get("downloaded", [])),
        "failed": len(progress.get("failed", [])),
    }

stats = get_current_stats()
print("=" * 50)
print("CURRENT DATASET STATISTICS")
print("=" * 50)
print(f"Total PDFs:      {stats['total_pdfs']:,}")
print(f"Total Size:      {stats['total_size_mb']:,.2f} MB")
print(f"Avg PDF Size:    {stats['avg_size_mb']:.2f} MB")
print(f"Progress file:   {stats['downloaded']} downloaded, {stats['failed']} failed")
print("=" * 50)
print(f"Remaining to target: {TARGET_PDF_COUNT - stats['total_pdfs']:,} PDFs")

CURRENT DATASET STATISTICS
Total PDFs:      2,739
Total Size:      12,172.45 MB
Avg PDF Size:    4.44 MB
Progress file:   2134 downloaded, 6 failed
Remaining to target: 7,261 PDFs


---

## 2. arXiv Search Queries

We use multiple search queries to find papers likely containing charts.

In [29]:
# ============================================================================
# ARXIV SEARCH QUERIES (EXPANDED)
# ============================================================================

ARXIV_QUERIES = [
    # ===================== COMPUTER VISION =====================
    'cat:cs.CV AND (chart OR visualization OR diagram)',
    'cat:cs.CV AND "bar chart"',
    'cat:cs.CV AND "line chart"', 
    'cat:cs.CV AND "pie chart"',
    'cat:cs.CV AND "scatter plot"',
    'cat:cs.CV AND "confusion matrix"',
    'cat:cs.CV AND "ROC curve"',
    'cat:cs.CV AND "precision recall"',
    'cat:cs.CV AND ablation',
    'cat:cs.CV AND benchmark',
    
    # ===================== MACHINE LEARNING =====================
    'cat:cs.LG AND (benchmark OR comparison OR performance)',
    'cat:cs.LG AND "accuracy" AND "training"',
    'cat:cs.LG AND ablation',
    'cat:cs.LG AND evaluation',
    'cat:cs.LG AND "loss curve"',
    'cat:cs.LG AND "training curve"',
    'cat:cs.LG AND hyperparameter',
    'cat:cs.LG AND "learning rate"',
    'cat:cs.LG AND convergence',
    'cat:cs.LG AND "model comparison"',
    
    # ===================== NLP =====================
    'cat:cs.CL AND benchmark',
    'cat:cs.CL AND evaluation',
    'cat:cs.CL AND performance',
    'cat:cs.CL AND "BLEU score"',
    'cat:cs.CL AND "F1 score"',
    'cat:cs.CL AND leaderboard',
    'cat:cs.CL AND comparison',
    
    # ===================== AI GENERAL =====================
    'cat:cs.AI AND benchmark',
    'cat:cs.AI AND evaluation',
    'cat:cs.AI AND "experimental results"',
    'cat:cs.AI AND comparison',
    
    # ===================== STATISTICS =====================
    'cat:stat.ML AND visualization',
    'cat:stat.ML AND "performance comparison"',
    'cat:stat.AP AND (chart OR graph)',
    'cat:stat.ME AND visualization',
    'cat:stat.CO AND figure',
    
    # ===================== ECONOMICS/FINANCE =====================
    'cat:econ.GN AND (chart OR figure)',
    'cat:q-fin.ST AND (chart OR visualization)',
    'cat:q-fin.TR AND "time series"',
    'cat:econ.EM AND (graph OR figure)',
    
    # ===================== PHYSICS (lots of plots) =====================
    'cat:physics.data-an AND visualization',
    'cat:astro-ph AND "light curve"',
    'cat:hep-ex AND "cross section"',
    'cat:cond-mat AND "phase diagram"',
    
    # ===================== BIOLOGY/MEDICAL =====================
    'cat:q-bio.QM AND visualization',
    'cat:cs.CE AND (chart OR plot)',
    
    # ===================== SPECIFIC CHART PAPERS =====================
    '"chart understanding"',
    '"chart question answering"',
    '"data visualization" AND extraction',
    '"figure understanding"',
    '"scientific figure"',
    '"plot digitization"',
    '"graph neural" AND visualization',
    
    # ===================== DEEP LEARNING BENCHMARKS =====================
    'cat:cs.LG AND ImageNet',
    'cat:cs.LG AND CIFAR',
    'cat:cs.LG AND "COCO dataset"',
    'cat:cs.CL AND GLUE',
    'cat:cs.CL AND SQuAD',
    'cat:cs.CV AND "object detection" AND benchmark',
    'cat:cs.CV AND segmentation AND benchmark',
    
    # ===================== SURVEY PAPERS (many figures) =====================
    'cat:cs.CV AND survey',
    'cat:cs.LG AND survey',
    'cat:cs.CL AND survey',
    'cat:cs.AI AND survey',
    
    # ===================== YEAR-SPECIFIC (more variety) =====================
    'cat:cs.LG AND submittedDate:[2024 TO 2025] AND results',
    'cat:cs.CV AND submittedDate:[2024 TO 2025] AND experiments',
    'cat:cs.CL AND submittedDate:[2024 TO 2025] AND evaluation',
    'cat:cs.LG AND submittedDate:[2023 TO 2024] AND benchmark',
    'cat:cs.CV AND submittedDate:[2023 TO 2024] AND comparison',
    
    # ===================== ADDITIONAL QUERIES =====================
    '"experimental evaluation"',
    '"quantitative results"',
    '"performance metrics"',
    '"baseline comparison"',
    '"state-of-the-art" AND results',
    'cat:cs.LG AND "Table 1"',  # Papers with tables often have charts
    'cat:cs.CV AND "Figure" AND "accuracy"',
]

print(f"Total search queries: {len(ARXIV_QUERIES)}")
print(f"Estimated max papers: {len(ARXIV_QUERIES) * 200:,} (with 200 per query)")
print(f"\nQueries by category:")
print(f"  - Computer Vision: 10")
print(f"  - Machine Learning: 10")
print(f"  - NLP: 7")
print(f"  - AI General: 4")
print(f"  - Statistics: 5")
print(f"  - Economics/Finance: 4")
print(f"  - Physics: 4")
print(f"  - Biology/Medical: 2")
print(f"  - Chart-specific: 7")
print(f"  - Deep Learning Benchmarks: 7")
print(f"  - Surveys: 4")
print(f"  - Year-specific: 5")
print(f"  - Additional: 7")

Total search queries: 76
Estimated max papers: 15,200 (with 200 per query)

Queries by category:
  - Computer Vision: 10
  - Machine Learning: 10
  - NLP: 7
  - AI General: 4
  - Statistics: 5
  - Economics/Finance: 4
  - Physics: 4
  - Biology/Medical: 2
  - Chart-specific: 7
  - Deep Learning Benchmarks: 7
  - Surveys: 4
  - Year-specific: 5
  - Additional: 7


---

## 3. ArXiv Hunter Class

Core class for searching and downloading papers.

In [30]:
# ============================================================================
# ARXIV HUNTER CLASS
# ============================================================================
import hashlib
import random
import time
from dataclasses import dataclass
from typing import List, Optional
from datetime import datetime

import requests

@dataclass
class ArxivPaper:
    """Represents an arXiv paper."""
    arxiv_id: str
    title: str
    authors: List[str]
    abstract: str
    published_date: datetime
    pdf_url: str
    categories: List[str]
    search_query: str = ""
    
    @property
    def safe_id(self) -> str:
        """Get filesystem-safe ID."""
        return self.arxiv_id.replace("/", "_").replace(".", "_")


class ArxivHunter:
    """Hunt and download papers from arXiv."""
    
    def __init__(self, output_dir: Path, rate_limit: float = 3.0):
        self.output_dir = output_dir
        self.rate_limit = rate_limit
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "ChartAnalysisResearch/1.0 (Academic Research)"
        })
        
        # Ensure arxiv library
        try:
            import arxiv
            self.arxiv = arxiv
        except ImportError:
            print("Installing arxiv library...")
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "arxiv", "-q"])
            import arxiv
            self.arxiv = arxiv
    
    def search(self, query: str, max_results: int = 100) -> List[ArxivPaper]:
        """Search arXiv for papers."""
        papers = []
        
        client = self.arxiv.Client(
            page_size=50,
            delay_seconds=3.0,
            num_retries=3,
        )
        
        search = self.arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=self.arxiv.SortCriterion.SubmittedDate,
        )
        
        try:
            for result in client.results(search):
                paper = ArxivPaper(
                    arxiv_id=result.entry_id.split("/")[-1],
                    title=result.title,
                    authors=[a.name for a in result.authors],
                    abstract=result.summary,
                    published_date=result.published,
                    pdf_url=result.pdf_url,
                    categories=result.categories,
                    search_query=query,
                )
                papers.append(paper)
        except Exception as e:
            print(f"Search error: {e}")
        
        return papers
    
    def download(self, paper: ArxivPaper) -> bool:
        """Download PDF for a paper."""
        pdf_path = self.output_dir / f"arxiv_{paper.safe_id}.pdf"
        
        if pdf_path.exists():
            return True  # Already downloaded
        
        try:
            response = self.session.get(paper.pdf_url, timeout=60)
            response.raise_for_status()
            
            pdf_path.write_bytes(response.content)
            
            # Rate limit
            time.sleep(self.rate_limit * random.uniform(0.8, 1.2))
            
            return True
        except Exception as e:
            print(f"Download failed | arxiv_id={paper.arxiv_id} | error={e}")
            return False


print("ArxivHunter class defined.")

ArxivHunter class defined.


---

## 4. Search Preview (Dry Run)

Preview search results without downloading.

In [24]:
# ============================================================================
# SEARCH PREVIEW
# ============================================================================

def preview_search(queries: List[str], papers_per_query: int = 5):
    """Preview search results without downloading."""
    hunter = ArxivHunter(RAW_PDFS_DIR)
    
    total_found = 0
    results_by_query = []
    
    for i, query in enumerate(queries[:3], 1):  # Only first 3 queries
        print(f"\n[{i}/{len(queries)}] Searching: {query[:50]}...")
        
        papers = hunter.search(query, max_results=papers_per_query)
        total_found += len(papers)
        
        results_by_query.append({
            "query": query,
            "count": len(papers),
            "sample_titles": [p.title[:60] for p in papers[:2]]
        })
        
        for paper in papers[:2]:
            print(f"  - {paper.title[:60]}...")
        
        time.sleep(2)  # Be nice to arXiv API
    
    print(f"\n" + "=" * 50)
    print(f"Preview complete: Found {total_found} papers from {len(queries[:3])} queries")
    print(f"Estimated total with all {len(ARXIV_QUERIES)} queries: ~{total_found * len(ARXIV_QUERIES) // 3}")
    
    return results_by_query

# Run preview
print("Running search preview (first 3 queries)...")
preview_results = preview_search(ARXIV_QUERIES, papers_per_query=5)

Running search preview (first 3 queries)...

[1/21] Searching: cat:cs.CV AND (chart OR visualization OR diagram)...
  - VisGym: Diverse, Customizable, Scalable Environments for Mul...
  - Reward-Forcing: Autoregressive Video Generation with Reward ...

[2/21] Searching: cat:cs.CV AND "bar chart"...
  - FinMMR: Make Financial Numerical Reasoning More Multimodal, ...
  - MapIQ: Evaluating Multimodal Large Language Models for Map Q...

[3/21] Searching: cat:cs.CV AND "line chart"...
  - Reclaiming the Horizon: Novel Visualization Designs for Time...
  - LineFormer: Rethinking Line Chart Data Extraction as Instanc...

Preview complete: Found 15 papers from 3 queries
Estimated total with all 21 queries: ~105


---

## 5. Batch Download with Progress Tracking

Download PDFs with checkpoint/resume capability.

In [31]:
# ============================================================================
# BATCH DOWNLOAD WITH PROGRESS
# ============================================================================

def load_progress() -> dict:
    """Load download progress from checkpoint."""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"downloaded": [], "failed": [], "last_query_idx": 0}


def save_progress(progress: dict):
    """Save download progress to checkpoint."""
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)


def reset_query_progress():
    """Reset query index to start from beginning (keeps downloaded list)."""
    if PROGRESS_FILE.exists():
        progress = load_progress()
        progress["last_query_idx"] = 0
        save_progress(progress)
        print(f"Reset query index to 0. Keeping {len(progress['downloaded'])} downloaded IDs.")
    else:
        print("No progress file found.")


def batch_download(
    target: int = 10000,
    papers_per_query: int = 200,
    resume: bool = True,
    dry_run: bool = True,
    start_query_idx: int = None,  # NEW: Override start index
):
    """
    Batch download arXiv papers.
    
    Args:
        target: Total papers to download
        papers_per_query: Max papers per search query
        resume: Whether to resume from checkpoint
        dry_run: If True, don't actually download
        start_query_idx: Override starting query index (useful when adding new queries)
    """
    hunter = ArxivHunter(RAW_PDFS_DIR)
    
    # Load or init progress
    if resume:
        progress = load_progress()
    else:
        progress = {"downloaded": [], "failed": [], "last_query_idx": 0}
    
    downloaded_ids = set(progress["downloaded"])
    success_count = len(downloaded_ids)
    
    # Override start index if provided
    if start_query_idx is not None:
        start_idx = start_query_idx
    else:
        start_idx = progress.get("last_query_idx", 0)
    
    print("=" * 60)
    print(f"BATCH DOWNLOAD | target={target} | mode={'DRY RUN' if dry_run else 'ACTIVE'}")
    print(f"Already downloaded: {success_count}")
    print(f"Starting from query index: {start_idx}/{len(ARXIV_QUERIES)}")
    print(f"Queries remaining: {len(ARXIV_QUERIES) - start_idx}")
    print("=" * 60)
    
    if dry_run:
        print("\n[DRY RUN] Would search and download papers...")
        print("Set EXECUTE_DOWNLOAD = True and dry_run=False to actually download.")
        return
    
    new_downloads = 0
    
    for query_idx, query in enumerate(ARXIV_QUERIES[start_idx:], start=start_idx):
        if success_count >= target:
            print(f"\n✓ Reached target of {target} papers!")
            break
        
        print(f"\n[Query {query_idx + 1}/{len(ARXIV_QUERIES)}] {query[:60]}...")
        
        try:
            papers = hunter.search(query, max_results=papers_per_query)
            new_in_query = sum(1 for p in papers if p.arxiv_id not in downloaded_ids)
            print(f"  Found {len(papers)} papers ({new_in_query} new)")
            
            for paper in papers:
                if success_count >= target:
                    break
                    
                if paper.arxiv_id in downloaded_ids:
                    continue
                
                success = hunter.download(paper)
                
                if success:
                    success_count += 1
                    new_downloads += 1
                    progress["downloaded"].append(paper.arxiv_id)
                    downloaded_ids.add(paper.arxiv_id)
                    
                    if new_downloads % 10 == 0:
                        print(f"  Progress: {success_count}/{target} (+{new_downloads} this session)")
                else:
                    progress["failed"].append(paper.arxiv_id)
                
                # Save checkpoint every 50 downloads
                if new_downloads % 50 == 0:
                    progress["last_query_idx"] = query_idx
                    save_progress(progress)
            
        except KeyboardInterrupt:
            print("\n[INTERRUPTED] Saving progress...")
            progress["last_query_idx"] = query_idx
            save_progress(progress)
            return
        
        except Exception as e:
            print(f"  Error: {e}")
            continue
    
    # Final save
    progress["last_query_idx"] = len(ARXIV_QUERIES)  # Mark all done
    save_progress(progress)
    
    print("\n" + "=" * 60)
    print("DOWNLOAD SESSION COMPLETE")
    print(f"  Total in collection: {success_count}")
    print(f"  New this session: {new_downloads}")
    print(f"  Failed: {len(progress['failed'])}")
    print("=" * 60)


print("batch_download() function defined.")
print("\nUsage:")
print("  batch_download(target=10000, dry_run=False)")
print("  batch_download(target=10000, dry_run=False, start_query_idx=20)  # Start from new queries")

batch_download() function defined.

Usage:
  batch_download(target=10000, dry_run=False)
  batch_download(target=10000, dry_run=False, start_query_idx=20)  # Start from new queries


In [32]:
# ============================================================================
# EXECUTE DOWNLOAD
# ============================================================================

if EXECUTE_DOWNLOAD:
    print("Starting batch download...")
    print("This will take several hours for 10,000 PDFs.")
    print("Press Ctrl+C to interrupt (progress will be saved).")
    print()
    
    # Start from query index 20 (where old queries ended, new queries begin)
    batch_download(
        target=TARGET_PDF_COUNT,
        papers_per_query=200,
        resume=True,
        dry_run=False,
        start_query_idx=20,  # Skip already-processed queries, start with new ones
    )
else:
    print("[SKIPPED] Download not executed.")
    print("Set EXECUTE_DOWNLOAD = True at the top to enable downloading.")
    print("\nTo download manually, run:")
    print("  batch_download(target=10000, dry_run=False, start_query_idx=20)")

Starting batch download...
This will take several hours for 10,000 PDFs.
Press Ctrl+C to interrupt (progress will be saved).

BATCH DOWNLOAD | target=10000 | mode=ACTIVE
Already downloaded: 2134
Starting from query index: 20/76
Queries remaining: 56

[Query 21/76] cat:cs.CL AND benchmark...
  Found 200 papers (0 new)

[Query 22/76] cat:cs.CL AND evaluation...
  Found 200 papers (0 new)

[Query 23/76] cat:cs.CL AND performance...
  Found 200 papers (0 new)

[Query 24/76] cat:cs.CL AND "BLEU score"...
  Found 200 papers (200 new)
  Progress: 2144/10000 (+10 this session)
  Progress: 2154/10000 (+20 this session)
  Progress: 2164/10000 (+30 this session)
  Progress: 2174/10000 (+40 this session)
Download failed | arxiv_id=2502.01882v2 | error=404 Client Error: Not Found for url: https://arxiv.org/pdf/2502.01882v2
  Progress: 2184/10000 (+50 this session)
Download failed | arxiv_id=2501.15405v2 | error=404 Client Error: Not Found for url: https://arxiv.org/pdf/2501.15405v2
  Progress: 2194

---

## 6. Download Report

Generate a report of the download progress.

In [33]:
# ============================================================================
# GENERATE DOWNLOAD REPORT
# ============================================================================

def generate_report():
    """Generate a summary report of the PDF collection."""
    stats = get_current_stats()
    
    # Get file dates
    pdf_files = list(RAW_PDFS_DIR.glob("*.pdf")) if RAW_PDFS_DIR.exists() else []
    
    if pdf_files:
        newest = max(pdf_files, key=lambda f: f.stat().st_mtime)
        oldest = min(pdf_files, key=lambda f: f.stat().st_mtime)
        newest_date = datetime.fromtimestamp(newest.stat().st_mtime)
        oldest_date = datetime.fromtimestamp(oldest.stat().st_mtime)
    else:
        newest_date = oldest_date = None
    
    report = f"""
================================================================================
                         PDF COLLECTION REPORT
                         {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
================================================================================

STATISTICS
----------
Total PDFs:           {stats['total_pdfs']:,}
Total Size:           {stats['total_size_mb']:,.2f} MB
Average PDF Size:     {stats['avg_size_mb']:.2f} MB

PROGRESS
--------
Target:               {TARGET_PDF_COUNT:,}
Downloaded:           {stats['downloaded']:,}
Failed:               {stats['failed']:,}
Remaining:            {max(0, TARGET_PDF_COUNT - stats['total_pdfs']):,}
Progress:             {stats['total_pdfs'] / TARGET_PDF_COUNT * 100:.1f}%

DATE RANGE
----------
Oldest PDF:           {oldest_date.strftime('%Y-%m-%d') if oldest_date else 'N/A'}
Newest PDF:           {newest_date.strftime('%Y-%m-%d') if newest_date else 'N/A'}

================================================================================
NEXT STEPS
================================================================================
1. Run 01b_image_extraction.ipynb to extract images from PDFs
2. Run 01c_chart_classification.ipynb to filter charts
3. Run 01d_qa_generation.ipynb to generate QA pairs
================================================================================
"""
    
    print(report)
    return stats


# Generate report
report_stats = generate_report()


                         PDF COLLECTION REPORT
                         2026-01-27 15:53:56

STATISTICS
----------
Total PDFs:           9,508
Total Size:           40,208.35 MB
Average PDF Size:     4.23 MB

PROGRESS
--------
Target:               10,000
Downloaded:           9,000
Failed:               51
Remaining:            492
Progress:             95.1%

DATE RANGE
----------
Oldest PDF:           2026-01-20
Newest PDF:           2026-01-27

NEXT STEPS
1. Run 01b_image_extraction.ipynb to extract images from PDFs
2. Run 01c_chart_classification.ipynb to filter charts
3. Run 01d_qa_generation.ipynb to generate QA pairs



---

## Summary

### Functions Provided

| Function | Purpose |
| --- | --- |
| `ArxivHunter.search()` | Search arXiv for papers |
| `ArxivHunter.download()` | Download PDF for a paper |
| `batch_download()` | Batch download with checkpoints |
| `generate_report()` | Generate collection statistics |

### Progress Tracking

- Progress is saved to `data/arxiv_progress.json`
- Can resume interrupted downloads
- Tracks both successful and failed downloads

### Rate Limiting

- Default: 3 seconds between downloads
- Respectful of arXiv API guidelines
- Random jitter to avoid patterns

---

**Next**: [01b_image_extraction.ipynb](01b_image_extraction.ipynb)